In [1]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd

index_personnes = Path("../../../corpus/IndexPersonnes.xml")

ns = "{http://www.tei-c.org/ns/1.0}"
xml_ns = "{http://www.w3.org/XML/1998/namespace}"

data = []

# Parsing en streaming
context = ET.iterparse(index_personnes, events=("end",))

for event, elem in context:
    if elem.tag == f"{ns}person":
        
        xml_id = elem.get(f"{xml_ns}id")
        role = elem.get("role")
        source = elem.get("source")

        # naissance
        birth = elem.find(f"{ns}birth")
        birth_date = birth.get("when") if birth is not None else None
        birth_place = birth.find(f"{ns}placeName").text if (birth is not None and birth.find(f"{ns}placeName") is not None) else None

        # mort
        death = elem.find(f"{ns}death")
        death_date = death.get("when") if death is not None else None
        death_place = death.find(f"{ns}placeName").text if (death is not None and death.find(f"{ns}placeName") is not None) else None

        data.append({
            "xml_id": xml_id,
            "role": role,
            "wikidata_source": source,
            "birth_date": birth_date,
            "birth_place": birth_place,
            "death_date": death_date,
            "death_place": death_place
        })

        # 🔥 TRÈS IMPORTANT : libérer la mémoire
        elem.clear()

# DataFrame
df = pd.DataFrame(data)

print(df.head())

df.to_csv("personnes.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '../../../corpus/IndexPersonnes.xml'

In [ ]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd
import time


index_personnes = Path("../../../corpus/IndexPersonnes.xml")
index_lieux = Path("../../../corpus/IndexPersonnes.xml")



In [7]:
import xml.etree.ElementTree as ET
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd
import requests
import time

# Namespaces
TEI_NS = "{http://www.tei-c.org/ns/1.0}"
XML_NS = "{http://www.w3.org/XML/1998/namespace}"

# Endpoint SPARQL Wikidata
SPARQL_URL = "https://query.wikidata.org/sparql"

headers = {
    "User-Agent": "TEI-Wikidata-Script/1.0 (pierre.husson.56@gmail.com)"
}

# ----------------------------
# 1. EXTRAIRE LES QID PERSONNES
# ----------------------------
def extract_person_qids(xml_file):
    qids = []

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}person":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                qids.append(qid)
            elem.clear()

    return list(set(qids))  # unique


# ----------------------------
# 2. SPARQL → LIEUX P19 / P20
# ----------------------------
def get_places_from_wikidata(qid):
    query = f"""
    SELECT ?birthPlace ?deathPlace WHERE {{
      OPTIONAL {{ wd:{qid} wdt:P19 ?birthPlace. }}
      OPTIONAL {{ wd:{qid} wdt:P20 ?deathPlace. }}
    }}
    """

    try:
        response = requests.get(
            SPARQL_URL,
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=10
        )

        # ✅ vérifier statut HTTP
        if response.status_code != 200:
            print(f"HTTP {response.status_code} pour {qid}")
            return []

        # ✅ vérifier contenu
        if not response.text.strip():
            print(f"Réponse vide pour {qid}")
            return []

        data = response.json()

    except Exception as e:
        print(f"Erreur réseau ou JSON avec {qid}: {e}")
        return []

    places = []

    for result in data.get("results", {}).get("bindings", []):
        if "birthPlace" in result:
            places.append(result["birthPlace"]["value"].split("/")[-1])
        if "deathPlace" in result:
            places.append(result["deathPlace"]["value"].split("/")[-1])

    return places


# ----------------------------
# 3. EXTRAIRE INDEX DES LIEUX
# ----------------------------
def extract_place_index(xml_file):
    place_ids = set()

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}place":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                place_ids.add(qid)
            elem.clear()

    return place_ids


# ----------------------------
# 4. PIPELINE PRINCIPAL
# ----------------------------
def main(person_xml, place_xml):

    print("Extraction des personnes...")
    person_qids = extract_person_qids(person_xml)

    print(f"{len(person_qids)} personnes trouvées")

    all_places = set()

    print("Requête Wikidata...")
    for i, qid in enumerate(person_qids):
        try:
            places = get_places_from_wikidata(qid)
            all_places.update(places)

            # ⚠️ éviter blocage Wikidata (rate limit)
            time.sleep(0.1)

        except Exception as e:
            print(f"Erreur avec {qid}: {e}")

    print(f"{len(all_places)} lieux récupérés depuis Wikidata")

    # Sauvegarde TXT
    with open("wikidata_places.txt", "w", encoding="utf-8") as f:
        for p in sorted(all_places):
            f.write(p + "\n")

    print("Extraction index des lieux TEI...")
    place_index = extract_place_index(place_xml)

    # Comparaison
    missing = all_places - place_index

    print(f"{len(missing)} lieux manquants dans l'index TEI")

    with open("missing_places.txt", "w", encoding="utf-8") as f:
        for m in sorted(missing):
            f.write(m + "\n")

index_personnes = Path("../../corpus/IndexPersonnes.xml")
index_lieux = Path("../../../corpus/IndexPersonnes.xml")

# ----------------------------
# LANCEMENT
# ----------------------------
if __name__ == "__main__":
    main(index_personnes, index_lieux)

Extraction des personnes...
1967 personnes trouvées
Requête Wikidata...
HTTP 429 pour Q651795
HTTP 429 pour Q2966521
HTTP 429 pour Q326548
HTTP 429 pour Q179256
HTTP 429 pour Q3098816
HTTP 429 pour Q318117
HTTP 429 pour Q314932
HTTP 429 pour Q744092
HTTP 429 pour Q3166555
HTTP 429 pour Q302723
HTTP 429 pour Q134189
HTTP 429 pour Q184530
HTTP 429 pour Q852011
HTTP 429 pour Q15802758
HTTP 429 pour Q161582
HTTP 429 pour Q434058
HTTP 429 pour Q333323
HTTP 429 pour Q785355
HTTP 429 pour Q1243321
HTTP 429 pour Q7780
HTTP 429 pour Q3086124
HTTP 429 pour Q3856455
HTTP 429 pour Q60199
HTTP 429 pour Q747
HTTP 429 pour Q3121786
HTTP 429 pour Q29436209
HTTP 429 pour Q21855521
HTTP 429 pour Q673168
HTTP 429 pour Q270658
HTTP 429 pour Q32857761
HTTP 429 pour Q2288622
HTTP 429 pour Q203829
HTTP 429 pour Q160538
HTTP 429 pour Q134929
HTTP 429 pour Q2255180
HTTP 429 pour Q3169711
HTTP 429 pour Q116240
HTTP 429 pour Q538350
HTTP 429 pour Q1433
HTTP 429 pour Q721349
HTTP 429 pour Q530515
HTTP 429 pour Q1

KeyboardInterrupt: 

In [ ]:
import xml.etree.ElementTree as ET
import os
from pathlib import Path
from collections import Counter
import requests
import time

# Namespaces
TEI_NS = "{http://www.tei-c.org/ns/1.0}"
XML_NS = "{http://www.w3.org/XML/1998/namespace}"

# Endpoint SPARQL Wikidata
SPARQL_URL = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "TEI-Wikidata-Script/1.0 (pierre.husson.56@gmail.com)"
}

# ----------------------------
# 1. EXTRAIRE LES QID PERSONNES
# ----------------------------
def extract_person_qids(xml_file):
    qids = []

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}person":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                qids.append(qid)
            elem.clear()

    return list(set(qids))  # unique


# ----------------------------
# 2. SPARQL AVEC RETRY + BACKOFF
# ----------------------------
def sparql_query(query, max_retries=5):
    """
    Exécute une requête SPARQL avec retry exponentiel en cas de 429.
    """
    wait = 5  # secondes d'attente initiale

    for attempt in range(max_retries):
        try:
            response = requests.get(
                SPARQL_URL,
                params={"query": query, "format": "json"},
                headers=headers,
                timeout=30
            )

            if response.status_code == 200:
                if not response.text.strip():
                    print("Réponse vide.")
                    return None
                return response.json()

            elif response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", wait))
                print(f"429 Rate limit — attente {retry_after}s (tentative {attempt + 1}/{max_retries})")
                time.sleep(retry_after)
                wait *= 2  # backoff exponentiel

            else:
                print(f"HTTP {response.status_code} — abandon")
                return None

        except requests.exceptions.Timeout:
            print(f"Timeout (tentative {attempt + 1}/{max_retries}) — attente {wait}s")
            time.sleep(wait)
            wait *= 2

        except Exception as e:
            print(f"Erreur inattendue : {e}")
            return None

    print("Nombre maximum de tentatives atteint.")
    return None


# ----------------------------
# 3. BATCH : P19 / P20 pour N personnes
# ----------------------------
def get_places_batch(qids, batch_size=50):
    """
    Récupère les lieux de naissance/décès pour une liste de QIDs en une seule requête SPARQL.
    Wikidata recommande des batchs de 50 max.
    """
    all_places = set()

    for i in range(0, len(qids), batch_size):
        batch = qids[i:i + batch_size]
        values = " ".join(f"wd:{qid}" for qid in batch)

        query = f"""
        SELECT ?person ?birthPlace ?deathPlace WHERE {{
          VALUES ?person {{ {values} }}
          OPTIONAL {{ ?person wdt:P19 ?birthPlace. }}
          OPTIONAL {{ ?person wdt:P20 ?deathPlace. }}
        }}
        """

        print(f"Batch {i // batch_size + 1} ({len(batch)} personnes)...")
        data = sparql_query(query)

        if data is None:
            print(f"Batch {i // batch_size + 1} ignoré suite à une erreur.")
            continue

        for result in data.get("results", {}).get("bindings", []):
            if "birthPlace" in result:
                all_places.add(result["birthPlace"]["value"].split("/")[-1])
            if "deathPlace" in result:
                all_places.add(result["deathPlace"]["value"].split("/")[-1])

        # Délai poli entre chaque batch (Wikidata recommande ~1s)
        time.sleep(1)

    return all_places


# ----------------------------
# 4. EXTRAIRE INDEX DES LIEUX TEI
# ----------------------------
def extract_place_index(xml_file):
    place_ids = set()

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}place":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                place_ids.add(qid)
            elem.clear()

    return place_ids


# ----------------------------
# 5. PIPELINE PRINCIPAL
# ----------------------------
def main(person_xml, place_xml):

    print("=" * 50)
    print("Extraction des personnes...")
    person_qids = extract_person_qids(person_xml)
    print(f"→ {len(person_qids)} personnes trouvées")

    print("\nRequête Wikidata (mode batch)...")
    all_places = get_places_batch(person_qids, batch_size=50)
    print(f"→ {len(all_places)} lieux récupérés depuis Wikidata")

    # Sauvegarde TXT
    with open("wikidata_places.txt", "w", encoding="utf-8") as f:
        for p in sorted(all_places):
            f.write(p + "\n")
    print("→ wikidata_places.txt sauvegardé")

    print("\nExtraction de l'index des lieux TEI...")
    place_index = extract_place_index(place_xml)
    print(f"→ {len(place_index)} lieux dans l'index TEI")

    # Comparaison
    missing = all_places - place_index
    print(f"\n→ {len(missing)} lieux manquants dans l'index TEI")

    with open("missing_places.txt", "w", encoding="utf-8") as f:
        for m in sorted(missing):
            f.write(m + "\n")
    print("→ missing_places.txt sauvegardé")
    print("=" * 50)


# ----------------------------
# CHEMINS
# ----------------------------
index_personnes = Path("../../corpus/IndexPersonnes.xml")
index_lieux = Path("../../corpus/IndexLieux.xml")  # ⚠️ corrigé (était IndexPersonnes)

if __name__ == "__main__":
    main(index_personnes, index_lieux)

Extraction des personnes...
→ 1967 personnes trouvées

Requête Wikidata (mode batch)...
Batch 1 (50 personnes)...
Batch 2 (50 personnes)...
Batch 3 (50 personnes)...
Batch 4 (50 personnes)...
Batch 5 (50 personnes)...
Batch 6 (50 personnes)...
Batch 7 (50 personnes)...
Batch 8 (50 personnes)...
Batch 9 (50 personnes)...
Batch 10 (50 personnes)...
Batch 11 (50 personnes)...
Batch 12 (50 personnes)...
Batch 13 (50 personnes)...
Batch 14 (50 personnes)...
Batch 15 (50 personnes)...
Batch 16 (50 personnes)...
Batch 17 (50 personnes)...
Batch 18 (50 personnes)...
Batch 19 (50 personnes)...
Batch 20 (50 personnes)...
Batch 21 (50 personnes)...
Batch 22 (50 personnes)...
Batch 23 (50 personnes)...
Batch 24 (50 personnes)...
Batch 25 (50 personnes)...
Batch 26 (50 personnes)...
Batch 27 (50 personnes)...
Batch 28 (50 personnes)...
Batch 29 (50 personnes)...
Batch 30 (50 personnes)...
Batch 31 (50 personnes)...
Batch 32 (50 personnes)...
Batch 33 (50 personnes)...
Batch 34 (50 personnes)...
Bat

FileNotFoundError: [Errno 2] No such file or directory: '../../../corpus/IndexLieux.xml'

In [10]:
import xml.etree.ElementTree as ET
import os
from pathlib import Path
from collections import Counter
import requests
import time

# Namespaces
TEI_NS = "{http://www.tei-c.org/ns/1.0}"
XML_NS = "{http://www.w3.org/XML/1998/namespace}"

# Endpoint SPARQL Wikidata
SPARQL_URL = "https://query.wikidata.org/sparql"

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "TEI-Wikidata-Script/1.0 (pierre.husson.56@gmail.com)"
}

# ----------------------------
# 1. EXTRAIRE LES QID + xml:id PERSONNES
# ----------------------------
def extract_person_qids(xml_file):
    """
    Retourne un dict { qid: xml_id } pour chaque <person> avec source Wikidata.
    """
    persons = {}  # qid -> xml:id

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}person":
            source = elem.get("source")
            xml_id = elem.get(f"{XML_NS}id", "unknown")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                persons[qid] = xml_id
            elem.clear()

    return persons  # { "Q123": "pers_001", ... }


# ----------------------------
# 2. SPARQL AVEC RETRY + BACKOFF
# ----------------------------
def sparql_query(query, max_retries=5):
    """
    Exécute une requête SPARQL avec retry exponentiel en cas de 429.
    """
    wait = 5  # secondes d'attente initiale

    for attempt in range(max_retries):
        try:
            response = requests.get(
                SPARQL_URL,
                params={"query": query, "format": "json"},
                headers=headers,
                timeout=30
            )

            if response.status_code == 200:
                if not response.text.strip():
                    print("Réponse vide.")
                    return None
                return response.json()

            elif response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", wait))
                print(f"429 Rate limit — attente {retry_after}s (tentative {attempt + 1}/{max_retries})")
                time.sleep(retry_after)
                wait *= 2  # backoff exponentiel

            else:
                print(f"HTTP {response.status_code} — abandon")
                return None

        except requests.exceptions.Timeout:
            print(f"Timeout (tentative {attempt + 1}/{max_retries}) — attente {wait}s")
            time.sleep(wait)
            wait *= 2

        except Exception as e:
            print(f"Erreur inattendue : {e}")
            return None

    print("Nombre maximum de tentatives atteint.")
    return None


# ----------------------------
# 3. BATCH : P19 / P20 pour N personnes
# ----------------------------
def get_places_batch(persons, output_file, batch_size=50):
    """
    Récupère les lieux de naissance/décès pour un dict { qid: xml_id }.
    Écrit les résultats dans output_file au fur et à mesure.
    Retourne un set de tous les QIDs de lieux trouvés.
    """
    all_places = set()
    qids = list(persons.keys())

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("xml_id\tpersonne_qid\ttype\tlieu_qid\n")  # en-tête

        for i in range(0, len(qids), batch_size):
            batch = qids[i:i + batch_size]
            values = " ".join(f"wd:{qid}" for qid in batch)

            query = f"""
            SELECT ?person ?birthPlace ?deathPlace WHERE {{
              VALUES ?person {{ {values} }}
              OPTIONAL {{ ?person wdt:P19 ?birthPlace. }}
              OPTIONAL {{ ?person wdt:P20 ?deathPlace. }}
            }}
            """

            print(f"Batch {i // batch_size + 1} ({len(batch)} personnes)...")
            data = sparql_query(query)

            if data is None:
                print(f"Batch {i // batch_size + 1} ignoré suite à une erreur.")
                continue

            for result in data.get("results", {}).get("bindings", []):
                person_qid = result["person"]["value"].split("/")[-1]
                xml_id = persons.get(person_qid, "unknown")

                if "birthPlace" in result:
                    lieu_qid = result["birthPlace"]["value"].split("/")[-1]
                    all_places.add(lieu_qid)
                    f.write(f"{xml_id}\t{person_qid}\tnaissance\t{lieu_qid}\n")
                    f.flush()  # écriture immédiate

                if "deathPlace" in result:
                    lieu_qid = result["deathPlace"]["value"].split("/")[-1]
                    all_places.add(lieu_qid)
                    f.write(f"{xml_id}\t{person_qid}\tdécès\t{lieu_qid}\n")
                    f.flush()

            # Délai poli entre chaque batch (Wikidata recommande ~1s)
            time.sleep(1)

    return all_places


# ----------------------------
# 4. EXTRAIRE INDEX DES LIEUX TEI
# ----------------------------
def extract_place_index(xml_file):
    place_ids = set()

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}place":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                place_ids.add(qid)
            elem.clear()

    return place_ids


# ----------------------------
# 5. PIPELINE PRINCIPAL
# ----------------------------
def main(person_xml, place_xml):

    print("=" * 50)
    print("Extraction des personnes...")
    persons = extract_person_qids(person_xml)
    print(f"→ {len(persons)} personnes trouvées")

    print("\nRequête Wikidata (mode batch)...")
    all_places = get_places_batch(persons, "wikidata_places.txt", batch_size=50)
    print(f"→ {len(all_places)} lieux récupérés depuis Wikidata")
    print("→ wikidata_places.txt sauvegardé (au fur et à mesure)")

    print("\nExtraction de l'index des lieux TEI...")
    place_index = extract_place_index(place_xml)
    print(f"→ {len(place_index)} lieux dans l'index TEI")
    with open("places.txt", "w", encoding="utf-8") as f:
        for element in place_index:
            f.write(element + "\n")

    # Comparaison
    missing = all_places - place_index
    print(f"\n→ {len(missing)} lieux manquants dans l'index TEI")

    with open("missing_places.txt", "w", encoding="utf-8") as f:
        for m in sorted(missing):
            f.write(m + "\n")
    print("→ missing_places.txt sauvegardé")
    print("=" * 50)


# ----------------------------
# CHEMINS
# ----------------------------
index_personnes = Path("../../corpus/IndexPersonnes.xml")
index_lieux = Path("../../corpus/IndexLieux.xml")  # ⚠️ corrigé (était IndexPersonnes)

if __name__ == "__main__":
    main(index_personnes, index_lieux)

Extraction des personnes...
→ 1967 personnes trouvées

Requête Wikidata (mode batch)...
Batch 1 (50 personnes)...
Batch 2 (50 personnes)...
Batch 3 (50 personnes)...
Batch 4 (50 personnes)...
Batch 5 (50 personnes)...
Batch 6 (50 personnes)...
Batch 7 (50 personnes)...
Batch 8 (50 personnes)...
Batch 9 (50 personnes)...
Batch 10 (50 personnes)...
Batch 11 (50 personnes)...
Batch 12 (50 personnes)...
Batch 13 (50 personnes)...
Batch 14 (50 personnes)...
Batch 15 (50 personnes)...
Batch 16 (50 personnes)...
Batch 17 (50 personnes)...
Batch 18 (50 personnes)...
Batch 19 (50 personnes)...
Batch 20 (50 personnes)...
Batch 21 (50 personnes)...
Batch 22 (50 personnes)...
Batch 23 (50 personnes)...
Batch 24 (50 personnes)...
Batch 25 (50 personnes)...
Batch 26 (50 personnes)...
Batch 27 (50 personnes)...
Batch 28 (50 personnes)...
Batch 29 (50 personnes)...
Batch 30 (50 personnes)...
Batch 31 (50 personnes)...
Batch 32 (50 personnes)...
Batch 33 (50 personnes)...
Batch 34 (50 personnes)...
Bat